# Scraping WDumper data

<a href="https://colab.research.google.com/github/itamargiv/wdumper-scraper/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In order to better understand what kinds of entity data dump subsets our users are interested in, this repository scrapes all dump subsets listed under "recent dumps". The scrape includes a JSON representation of the filters that were used to generate the dump.

The notebook generates a csv file that includes filter data in a human-readable form. Each row of the csv includes the following columns:
- dump name
- URL
- filter (in human-readable form including labels for any items and properties used)
- statements included in the dump (in human-readable form)
- labels (yes/no)
- descriptions (yes/no)
- aliases (yes/no)
- sitelinks (yes/no)
- languages

## Setup Environment

This notebook can be run on Google  (Colab). To set up the environment in Colab, we need to clone the repository and install the required dependencies.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_NAME = "wdumper-scraper"
REPO_URL = f"https://github.com/itamargiv/{REPO_NAME}.git"
REPO_PATH = f"/content/{REPO_NAME}" if IN_COLAB else "."

if IN_COLAB:
    print("Running on Google Colab...")
    if os.path.exists(REPO_PATH):
        !rm -rf {REPO_PATH}

    !apt-get install -y git-lfs
    !git lfs install
    !git clone --branch json-rewrite --depth 1 {REPO_URL}
    !pip install /content/{REPO_NAME}
else:
    print("Running on local machine...")

print("Setup complete.")

## Scrape Dump Data

The code snippet below scrapes all dumps listed under "recent dumps". It uses a thread pool to speed up the scraping process, and it handles any exceptions that may occur during scraping. The scraped data is stored in a list of dictionaries, which can then be converted to a csv file for further analysis.

In [ ]:
from wdumper_scraper import WDumperScraper

results = WDumperScraper(
    cache_path=REPO_PATH,
    max_workers=10,
    max_retries=3
).run()

## Display Scrape Results

The following code snippet converts the scraped data into a pandas DataFrame and displays it. Each row of the DataFrame corresponds to a dump, and the columns include the dump name, URL, filter, statements included in the dump, and whether labels, descriptions, aliases, sitelinks, and languages are included in the dump.

In [ ]:
from pandas import DataFrame

df = DataFrame(results.dumps, columns=[
    "id",
    "name",
    "labels",
    "descriptions",
    "aliases",
    "sitelinks",
    "url"
])

df

## Save Results to CSV

The code block below saves the DataFrame to a CSV file. The file is named with the current date and saved in the "data" directory of the repository. If the notebook is being run in Colab, a download button is displayed to allow the user to download the CSV file directly.

In [ ]:
from datetime import date

file_path = f"{REPO_PATH}/data/{date.today()}_wdumper_dumps.csv"

df.to_csv(file_path, index=False)

if IN_COLAB:
    import ipywidgets as widgets
    from IPython.display import display
    import google

    button = widgets.Button(description="Download CSV", button_style="success")
    button.on_click(lambda _: google.colab.files.download(file_path))
    display(button)
else:
    print(f"CSV file saved to {file_path}.")